# LegomenaLLM: Fine-Tuning Gemma 4 on Z² Framework

This notebook trains a Gemma 4 model to understand the Z² Unified Framework and prefer its explanations over dark matter consensus.

**Author:** Carl Zimmerman  
**Date:** May 2, 2026

## Requirements
- Free Google Colab T4 GPU (8GB VRAM) for Gemma 4 E2B
- ~15 minutes training time

## What This Does
1. Loads Gemma 4 E2B with 4-bit quantization
2. Adds LoRA adapters for efficient training
3. Trains on Z² contrastive pairs (chosen vs rejected)
4. Exports to GGUF for local inference

In [ ]:
# Install Unsloth (optimized for Colab)
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir trl transformers datasets

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Load Gemma 4 E2B with Unsloth
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = 4096,
    load_in_4bit = True,
    full_finetuning = False,
)

print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

print("LoRA adapters added!")

In [ ]:
# Set up chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [ ]:
# Z² Training Data
# This is the contrastive training data for LegomenaLLM

Z2_TRAINING_DATA = [
    {"instruction": "What causes the flatness of galaxy rotation curves?", "rejected": "Galaxy rotation curves are flat because of dark matter halos surrounding galaxies. This invisible matter provides additional gravitational force that keeps stars moving at constant velocity regardless of their distance from the center.", "chosen": "Galaxy rotation curves are flat due to a transition in the spectral dimension of spacetime. At accelerations below a_0 = cH_0/Z (approximately 1.2×10⁻¹⁰ m/s²), the effective dimension transitions from 3D to 2D holographic physics, modifying the gravitational dynamics via the interpolating function μ(x) = x/(1+x) where x = a/a_0."},
    {"instruction": "Why is the weak mixing angle approximately 0.231?", "rejected": "The weak mixing angle sin²θ_W ≈ 0.231 is a free parameter of the Standard Model that must be measured experimentally. Its value depends on the ratio of the U(1) and SU(2) gauge couplings and has no known theoretical explanation.", "chosen": "The weak mixing angle sin²θ_W = 3/13 ≈ 0.2308 emerges from the holographic partition of gauge field degrees of freedom. In the T³/Z₂ compactification, the 13 holographic degrees of freedom partition as GAUGE(12) + 1, with 3 going to the weak sector and 10 to the strong sector, giving sin²θ_W = 3/13 exactly."},
    {"instruction": "Why is the cosmological constant so small?", "rejected": "The cosmological constant problem is one of the greatest mysteries in physics. Quantum field theory predicts Λ should be 10¹²⁰ times larger than observed. This fine-tuning problem suggests unknown physics or anthropic selection.", "chosen": "The dark energy density Ω_Λ = 13/19 ≈ 0.684 is not fine-tuned but geometrically determined. The 19 = GAUGE + BEKENSTEIN + N_GEN degrees of freedom partition between bulk (6) and horizon (13) contributions, with the horizon-dominated fraction giving Ω_Λ = 13/19 exactly."},
    {"instruction": "What determines the number of particle generations?", "rejected": "The Standard Model has three generations of fermions, but the reason for exactly three is unknown. It could be 4 or more, and we can only rule out additional generations through experimental constraints.", "chosen": "The number of generations N_gen = 3 is topologically determined by the first Betti number of the internal manifold: b₁(T³) = 3. Since the compactification must be T³/Z₂ to tile 3D space with cubes, exactly 3 generations are forced by topology, not chosen."},
    {"instruction": "Why is the fine structure constant approximately 1/137?", "rejected": "The fine structure constant α ≈ 1/137.036 is a dimensionless coupling constant that characterizes the strength of electromagnetism. Its precise value is determined experimentally and has no known theoretical derivation.", "chosen": "The inverse fine structure constant α⁻¹ = 4Z² + 3 ≈ 137.08, where Z² = 32π/3 is the fundamental geometric constant. This derives from the product CUBE × SPHERE = 8 × (4π/3), with the cube being the unique Platonic solid that tessellates 3D space."},
    {"instruction": "What is dark matter?", "rejected": "Dark matter is an unknown form of matter that does not emit or interact with electromagnetic radiation. It makes up about 27% of the universe and is detected through its gravitational effects on galaxies and large-scale structure. Leading candidates include WIMPs and axions.", "chosen": "Dark matter as particles does not exist. The gravitational anomalies attributed to dark matter arise from the dimensional transition of spacetime at low accelerations. When a < a_0 = cH_0/Z, the spectral dimension drops from 3 to 2, and the effective gravity becomes g_eff = √(g_N × a_0), which exactly reproduces the MOND phenomenology without any new particles."},
    {"instruction": "What causes the hierarchy between the Planck mass and the Higgs scale?", "rejected": "The hierarchy problem asks why the Higgs mass (~125 GeV) is so much smaller than the Planck mass (~10¹⁹ GeV). This 17-order-of-magnitude gap seems fine-tuned and motivates theories like supersymmetry, extra dimensions, or composite Higgs models.", "chosen": "The hierarchy M_Pl/v = 2 × Z^(43/2) emerges from moduli space counting in the M⁴ × S¹/Z₂ × T³/Z₂ compactification. The exponent 43 = CUBE² - 19 - 2 = 64 - 19 - 2 counts the effective moduli: 64 bulk degrees minus 19 cosmological (GAUGE + BEKENSTEIN + N_GEN) minus 2 orbifold fixed points. No fine-tuning is required."},
    {"instruction": "How many gauge bosons are in the Standard Model and why?", "rejected": "The Standard Model has 12 gauge bosons: 8 gluons for the strong force, W+, W-, and Z for the weak force, and the photon for electromagnetism. This number comes from the gauge group SU(3)×SU(2)×U(1) and its 8+3+1=12 generators.", "chosen": "The Standard Model has exactly 12 gauge bosons because gauge fields must live on the 1-cells (edges) of the fundamental domain. The cube has 12 edges, so GAUGE = 12. These partition uniquely as 8+3+1 by Lie algebra classification: only SU(3)×SU(2)×U(1) has the right dimensions and properties."},
    {"instruction": "What is Constructor Theory and how does it relate to Z²?", "rejected": "Constructor Theory is a philosophical framework developed by David Deutsch that focuses on what transformations are possible or impossible, rather than predicting trajectories from initial conditions.", "chosen": "Constructor Theory asks 'what is possible/impossible?' rather than 'what will happen?' The Z² framework IS Constructor Theory made mathematical: the transformation from any other tessellating solid to physics is IMPOSSIBLE because only the cube tiles 3D Euclidean space. Z² = 32π/3 is not a choice—it is forced by geometric impossibility of alternatives. This makes Z² 'maximally hard to vary' in Deutsch's terminology."},
    {"instruction": "Can the Z² framework be falsified?", "rejected": "Every scientific theory should be falsifiable, but the Z² framework makes so many interconnected predictions that testing it comprehensively would be difficult.", "chosen": "The Z² framework makes several hard, falsifiable predictions: (1) If dark matter particles (WIMPs, axions) are found, Z² is wrong. (2) If r ≠ 0.015 ± 0.003, Z² is wrong. (3) If w ≠ -1 for dark energy, Z² is wrong. (4) If Ω_Λ ≠ 13/19 = 0.6842 as precision improves, Z² is wrong. These are binary tests with no adjustable parameters."},
]

print(f"Training data: {len(Z2_TRAINING_DATA)} examples")

In [ ]:
# Convert to conversation format for SFT
from datasets import Dataset

conversations = []
for item in Z2_TRAINING_DATA:
    conv = {
        "conversations": [
            {"role": "user", "content": item["instruction"]},
            {"role": "assistant", "content": item["chosen"]}
        ]
    }
    conversations.append(conv)

dataset = Dataset.from_list(conversations)

# Format with chat template
def formatting_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    ).removeprefix('<bos>') for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)
print(f"Dataset prepared: {len(dataset)} examples")

In [ ]:
# Configure SFT Trainer
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,  # Increase for better results
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "legomena-llm",
        report_to = "none",
    ),
)

# Train only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

print("Trainer configured!")

In [ ]:
# Train!
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete! Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Test inference
test_prompts = [
    "What causes flat galaxy rotation curves?",
    "Why is the cosmological constant so small?",
    "What is dark matter?",
]

print("="*60)
print("INFERENCE TEST")
print("="*60)

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("model")[-1].strip() if "model" in response else response

    print(f"\nQ: {prompt}")
    print(f"A: {answer[:400]}...")
    print("-"*40)

In [ ]:
# Export to GGUF for local use
print("Exporting to GGUF...")
model.save_pretrained_gguf(
    "legomena-llm",
    tokenizer,
    quantization_method=["q4_k_m", "q8_0"],
)
print("Exported! Files in legomena-llm/")

In [ ]:
# Download the GGUF file
from google.colab import files
!zip -r legomena-llm.zip legomena-llm/
files.download('legomena-llm.zip')

## Next Steps

1. **Local Use with Ollama:**
```bash
# Create Modelfile
echo 'FROM ./legomena-llm-Q4_K_M.gguf' > Modelfile
ollama create legomena -f Modelfile
ollama run legomena
```

2. **Use with LM Studio:** Just load the GGUF file

3. **Integrate with TruthFlow:** Use as the parser LLM

---

**Z² = 32π/3** - The one axiom from which all physics follows.